# Transformer の基礎

Self-Attention だけで構成されたモデル。LSTM を完全に排除。

```
RNN → LSTM → Attention → Transformer
 ✅     ✅       ✅         ★ 今ここ
```

## 学習の流れ
1. Positional Encoding（位置情報）
2. Multi-Head Attention（複数の注意）
3. Transformer Encoder / Decoder
4. 系列反転タスクで学習・評価
5. Attention の前回との比較

## 全体構造

```
         Encoder                              Decoder
  ┌─────────────────┐                ┌──────────────────────┐
  │                 │                │                      │
  │  Self-Attention │                │  Masked Self-Attn    │
  │  + Residual     │                │  + Residual          │
  │  + LayerNorm    │                │  + LayerNorm         │
  │       ↓         │                │       ↓              │
  │  Feed-Forward   │───encoder──→   │  Cross-Attention     │
  │  + Residual     │   output       │  (Enc-Dec Attention) │
  │  + LayerNorm    │                │  + Residual          │
  │                 │                │  + LayerNorm         │
  └─────────────────┘                │       ↓              │
        × N層                        │  Feed-Forward        │
                                     │  + Residual          │
                                     │  + LayerNorm         │
                                     └──────────────────────┘
                                           × N層
```

前回との違い:
- LSTM → **Self-Attention** に置き換え
- 新要素: **Positional Encoding**, **Multi-Head**, **Residual + LayerNorm**, **Causal Mask**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print('PyTorch version:', torch.__version__)

## 1. Positional Encoding（位置情報）

Self-Attention は全要素を同時に見るので、**語順がわからない**。

```
"I love cats" と "cats love I" が同じに見えてしまう
```

解決策: 各位置に固有の**位置ベクトル**を足す。

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

- 位置ごとに異なるパターンの sin/cos を使う
- 偶数次元は sin、奇数次元は cos
- 学習不要（固定値）

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数次元: sin
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数次元: cos
        self.register_buffer('pe', pe.unsqueeze(0))    # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1)]


# === 可視化 ===
pe_module = PositionalEncoding(d_model=64)
pe_values = pe_module.pe.squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 左: ヒートマップ
im = axes[0].imshow(pe_values[:20, :].T, cmap='RdBu', aspect='auto')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
axes[0].set_title('Positional Encoding (20 positions x 64 dims)')
plt.colorbar(im, ax=axes[0])

# 右: 各次元の波形
for dim in [0, 1, 4, 5, 10, 11]:
    axes[1].plot(pe_values[:20, dim], label=f'dim {dim}')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].set_title('PE values per dimension')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transformer_positional_encoding.png', dpi=150, bbox_inches='tight')
plt.show()

print('各位置が異なるパターンを持つ → 位置を区別できる')
print('低い次元ほど高い周波数（細かい変化）、高い次元ほど低い周波数（ゆるやかな変化）')

## 2. Multi-Head Attention

前回の Self-Attention は **1つの視点** でしか関連度を計算できなかった。

Multi-Head は **複数の視点を並列で持つ**:

```
Single Head (前回):  64次元で1回の Attention

Multi-Head (4 heads): 64次元 → 16次元 × 4つに分割
  Head 0: 16次元の QKV → Attention → 16次元の出力（文法的関係を学ぶかも）
  Head 1: 16次元の QKV → Attention → 16次元の出力（意味的関係を学ぶかも）
  Head 2: 16次元の QKV → Attention → 16次元の出力（位置関係を学ぶかも）
  Head 3: 16次元の QKV → Attention → 16次元の出力（別の関係を学ぶかも）
  → 結合 → 64次元に戻す
```

各 Head が異なる種類の関係を独立に学習できる。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # 各ヘッドの次元数

        self.W_q = nn.Linear(d_model, d_model)  # 全ヘッド分まとめて変換
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)  # 結合後の変換

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # --- ① 線形変換 + ヘッドに分割 ---
        # (batch, seq_len, d_model) → (batch, n_heads, seq_len, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        # --- ② Scaled Dot-Product Attention（各ヘッド独立）---
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)

        # --- ③ ヘッドを結合 ---
        # (batch, n_heads, seq_len, d_k) → (batch, seq_len, d_model)
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k)
        output = self.W_o(output)

        return output, attention_weights


# 動作確認
mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(1, 6, 64)  # (batch=1, seq_len=6, d_model=64)
out, weights = mha(x, x, x)  # Self-Attention: Q=K=V=x
print(f'Input:  {list(x.shape)}')
print(f'Output: {list(out.shape)}')
print(f'Attention weights: {list(weights.shape)}  (batch, heads, seq, seq)')
print(f'  → 4つのヘッドがそれぞれ 6x6 の Attention weights を持つ')

## 3. Transformer の各層

### Residual Connection（残差接続）

```
出力 = LayerNorm(x + Attention(x))
                 ↑
                 元の入力をそのまま足す
```

層が深くなっても勾配が消えにくくなる（「x をそのまま通すショートカット」がある）。

### Causal Mask（因果マスク）

Decoder の Self-Attention で**未来の情報を見えなくする**マスク。

```
Decoder 入力: [SOS, 9, 5, 1, 4, 1]

SOS を見て 9 を予測したい → SOS だけ見える
 9  を見て 5 を予測したい → SOS, 9 だけ見える
 5  を見て 1 を予測したい → SOS, 9, 5 だけ見える
 ...

マスク:
  [1, 0, 0, 0, 0, 0]  ← 位置0: SOS だけ見える
  [1, 1, 0, 0, 0, 0]  ← 位置1: SOS と 9
  [1, 1, 1, 0, 0, 0]  ← 位置2: SOS, 9, 5
  [1, 1, 1, 1, 0, 0]
  [1, 1, 1, 1, 1, 0]
  [1, 1, 1, 1, 1, 1]  ← 位置5: 全部見える

0 の位置 → スコアを -∞ にする → softmax で 0 になる
```

これにより **全位置を並列に計算しつつ、各位置は過去しか見えない** 状態を実現。
LSTM のように1ステップずつ処理する必要がなくなる。

In [ ]:
class FeedForward(nn.Module):
    """位置ごとの全結合層（各位置を独立に変換）"""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # Self-Attention + Residual + LayerNorm
        attn_out, _ = self.self_attention(x, x, x, mask)
        x = self.norm1(x + attn_out)
        # Feed-Forward + Residual + LayerNorm
        ff_out = self.feed_forward(x)
        x = self.norm2(x + ff_out)
        return x


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads)
        self.cross_attention = MultiHeadAttention(d_model, n_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, tgt_mask=None):
        # ① Masked Self-Attention（出力系列内の関係、未来は見えない）
        attn_out, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + attn_out)
        # ② Cross-Attention（出力から入力を参照 = 前回の Enc-Dec Attention）
        attn_out, attn_weights = self.cross_attention(x, encoder_output, encoder_output)
        x = self.norm2(x + attn_out)
        # ③ Feed-Forward
        ff_out = self.feed_forward(x)
        x = self.norm3(x + ff_out)
        return x, attn_weights


print('Encoder Layer: Self-Attention → Feed-Forward')
print('Decoder Layer: Masked Self-Attention → Cross-Attention → Feed-Forward')
print('各層に Residual + LayerNorm がつく')

In [ ]:
# === 定数 ===
VOCAB_SIZE = 10
SOS_TOKEN = 10
NUM_TOKENS = 11
SEQ_LEN = 6

D_MODEL = 64
N_HEADS = 4
D_FF = 128
N_LAYERS = 2
BATCH_SIZE = 64


class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def make_causal_mask(self, size):
        """未来を見えなくする下三角マスク"""
        mask = torch.tril(torch.ones(size, size)).unsqueeze(0).unsqueeze(0)
        return mask  # (1, 1, size, size)

    def encode(self, src):
        x = self.pos_encoding(self.embedding(src) * (self.d_model ** 0.5))
        for layer in self.encoder_layers:
            x = layer(x)
        return x

    def decode(self, trg, enc_output, tgt_mask=None):
        x = self.pos_encoding(self.embedding(trg) * (self.d_model ** 0.5))
        attn_weights = None
        for layer in self.decoder_layers:
            x, attn_weights = layer(x, enc_output, tgt_mask)
        return self.fc_out(x), attn_weights

    def forward(self, src, trg):
        enc_output = self.encode(src)
        tgt_mask = self.make_causal_mask(trg.size(1))
        return self.decode(trg, enc_output, tgt_mask)


model = Transformer(NUM_TOKENS, D_MODEL, N_HEADS, D_FF, N_LAYERS)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

# Causal Mask の可視化
mask = model.make_causal_mask(SEQ_LEN).squeeze()
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(mask, cmap='Blues')
ax.set_title('Causal Mask')
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
for i in range(SEQ_LEN):
    for j in range(SEQ_LEN):
        ax.text(j, i, int(mask[i, j].item()), ha='center', va='center')
plt.tight_layout()
plt.savefig('transformer_causal_mask.png', dpi=150, bbox_inches='tight')
plt.show()
print('1 = 見える, 0 = 見えない（-∞ でマスク）')

## 4. 学習

前回と同じ**系列反転タスク**で学習します。

### LSTM+Attention との大きな違い

```python
# LSTM+Attention（前回）: Decoder を1ステップずつループ
for t in range(trg_len):
    prediction = decoder(input_t, hidden, cell, enc_out)

# Transformer: 全ステップを一括処理（Causal Mask で未来を隠す）
output = model(src, decoder_input)  # ← ループなし！
```

Transformer は**並列計算**ができるため、学習が高速。

In [ ]:
# === データ生成 ===
def generate_data(n_samples, seq_len):
    src = torch.randint(0, VOCAB_SIZE, (n_samples, seq_len))
    trg = src.flip(dims=[1])
    return src, trg

train_src, train_trg = generate_data(3000, SEQ_LEN)
test_src, test_trg = generate_data(500, SEQ_LEN)
print(f'Example: {train_src[0].tolist()} -> {train_trg[0].tolist()}')

# === 学習 ===
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(100):
    model.train()
    epoch_loss = 0
    n_batches = 0

    indices = torch.randperm(len(train_src))
    for i in range(0, len(train_src), BATCH_SIZE):
        batch_idx = indices[i:i+BATCH_SIZE]
        src = train_src[batch_idx]
        trg = train_trg[batch_idx]

        # Decoder 入力: [SOS, trg[0], trg[1], ..., trg[-2]]
        # 正解:         [trg[0], trg[1], ..., trg[-1]]
        decoder_input = torch.cat([
            torch.full((src.size(0), 1), SOS_TOKEN, dtype=torch.long),
            trg[:, :-1]
        ], dim=1)

        optimizer.zero_grad()
        output, _ = model(src, decoder_input)
        loss = criterion(output.reshape(-1, NUM_TOKENS), trg.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)

    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}/100  Loss: {avg_loss:.4f}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Transformer Training Loss')
plt.grid(True, alpha=0.3)
plt.savefig('transformer_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === 評価（自己回帰: 1トークンずつ生成）===
model.eval()
correct_count = 0
n_eval = 100
first_attn = None
first_src = None
first_pred = None

with torch.no_grad():
    for i in range(n_eval):
        src = test_src[i:i+1]
        trg = test_trg[i:i+1]

        enc_output = model.encode(src)

        # SOS から始めて1トークンずつ生成
        decoder_input = torch.full((1, 1), SOS_TOKEN, dtype=torch.long)
        predicted = []

        for t in range(SEQ_LEN):
            tgt_mask = model.make_causal_mask(decoder_input.size(1))
            output, attn_w = model.decode(decoder_input, enc_output, tgt_mask)
            next_token = output[:, -1, :].argmax(dim=-1).item()
            predicted.append(next_token)
            decoder_input = torch.cat([
                decoder_input,
                torch.tensor([[next_token]])
            ], dim=1)

        target = trg.squeeze().tolist()
        if predicted == target:
            correct_count += 1

        if i < 5:
            mark = 'OK' if predicted == target else 'NG'
            print(f'Input:  {src.squeeze().tolist()}')
            print(f'Target: {target}')
            print(f'Pred:   {predicted}  [{mark}]')
            print()

        if i == 0:
            first_attn = attn_w.squeeze().numpy()
            first_src = src.squeeze().tolist()
            first_pred = predicted

print(f'Sequence Accuracy: {correct_count}/{n_eval} = {correct_count/n_eval*100:.1f}%')

In [ ]:
# === Attention Heatmap ===
# Cross-Attention の重みを可視化（Multi-Head なので Head ごとに表示）
n_heads = first_attn.shape[0]
# decoder_input は [SOS, t0, t1, ..., t4] の7トークン
# 最後の6ステップ分（実際の予測部分）を使う
attn_to_plot = first_attn[:, -SEQ_LEN:, :]  # (n_heads, 6, 6)

fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
src_labels = [str(x) for x in first_src]
pred_labels = [str(x) for x in first_pred]

for h in range(n_heads):
    ax = axes[h]
    im = ax.imshow(attn_to_plot[h], cmap='Blues', aspect='auto')
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels(src_labels, fontsize=10)
    ax.set_yticklabels(pred_labels, fontsize=10)
    ax.set_xlabel('Input')
    ax.set_title(f'Head {h}')
    for i in range(SEQ_LEN):
        for j in range(SEQ_LEN):
            ax.text(j, i, f'{attn_to_plot[h, i, j]:.2f}',
                    ha='center', va='center', fontsize=7)

axes[0].set_ylabel('Output')
plt.suptitle('Cross-Attention Weights (per Head)', fontsize=14)
plt.tight_layout()
plt.savefig('transformer_attention_heads.png', dpi=150, bbox_inches='tight')
plt.show()

print('各 Head が異なるパターンの Attention を学習している')
print('反転タスクなので、少なくとも1つの Head に逆対角線パターンが見えるはず')

## まとめ

### 前回（LSTM+Attention）との比較

| | LSTM+Attention | Transformer |
|---|---|---|
| 系列の処理 | 順番に1つずつ | 全位置を並列処理 |
| 語順の理解 | LSTM が自然に持つ | Positional Encoding で付与 |
| Attention | 1つの Head | Multi-Head（複数の視点） |
| Decoder の学習 | 1ステップずつループ | Causal Mask で一括処理 |
| 長距離の依存関係 | 苦手（情報が薄れる） | 得意（全位置を直接参照） |

### Transformer の構成要素

| 要素 | 役割 |
|------|------|
| Positional Encoding | 語順情報の付与 |
| Multi-Head Attention | 複数の視点で関連度を計算 |
| Causal Mask | 未来の情報を隠す |
| Residual + LayerNorm | 深い層でも勾配を安定させる |
| Feed-Forward | 各位置の特徴を非線形変換 |

これが GPT（Decoder のみ）や BERT（Encoder のみ）の基盤です。

```
RNN → LSTM → Attention → Transformer
 ✅     ✅       ✅           ✅
```